# 114. Voronoi分割: マルチアサイン(assign=2)の効果

## 目的
- 画像プロジェクトと同じ方式（各ドキュメントを最近傍2セントロイドに重複登録）をテキストEmbeddingに適用
- NB111-113のassign=1（1セントロイドのみ）との比較

## 方式の違い
| | assign=1 (NB111-113) | assign=2 (本実験) |
|---|---|---|
| ドキュメント | 最近傍1つのpivot_id | 最近傍**2つ**のpivot_id |
| 候補の拾い方 | 1箇所のみ | 2箇所に存在→拾われやすい |
| Firestoreクエリ | `WHERE pivot_id IN [...]` | `WHERE pivot_ids array-contains-any [...]` |
| ストレージ | pivot_id: 整数1つ | pivot_ids: 整数配列[2] |

## 画像プロジェクト参考値
- C=256, assign=2, top_search=5 → 候補~3,200 (4.3%), BF-R@30=83.3%

## NB113参考値（assign=1）
- C=256, P=5: EN R@10=81.0% (候補3.4%), JA R@10=88.2% (候補3.5%)
- C=256, P=2: EN R@10=67.4% (候補1.4%), JA R@10=73.4% (候補1.5%)

## 0. セットアップ

In [1]:
import sys
import numpy as np
import time
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
np.random.seed(42)

N_QUERIES = 200
TOP_K = 10
print(f'Configuration: N_QUERIES={N_QUERIES}, TOP_K={TOP_K}')

Configuration: N_QUERIES=200, TOP_K=10


## 1. データロードとk-meansモデル構築

In [2]:
emb_en = np.load(DATA_DIR / '10k_e5_base_en_embeddings.npy')
emb_ja = np.load(DATA_DIR / '10k_e5_base_ja_embeddings.npy')

datasets = {'EN': emb_en, 'JA': emb_ja}
for name, emb in datasets.items():
    print(f'{name}: {emb.shape}')

# k-meansモデルを事前構築（NB113と同じ）
cluster_range = [32, 64, 128, 256]
kmeans_cache = {}

for ds_name, emb in datasets.items():
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    emb_normed = emb / norms
    kmeans_cache[ds_name] = {'emb_normed': emb_normed}
    
    print(f'\n{ds_name}: k-means構築中...')
    for n_c in cluster_range:
        kmeans = MiniBatchKMeans(n_clusters=n_c, random_state=42, batch_size=2048, n_init=3)
        kmeans.fit(emb_normed)
        centroids = kmeans.cluster_centers_
        centroids_normed = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
        
        # 全ドキュメントとセントロイドの類似度行列
        all_sims = emb_normed @ centroids_normed.T  # (N, C)
        
        kmeans_cache[ds_name][n_c] = {
            'centroids': centroids_normed,
            'all_sims': all_sims,
        }
        print(f'  C={n_c}: done')

EN: (10000, 768)
JA: (9990, 768)

EN: k-means構築中...


  C=32: done


  C=64: done


  C=128: done


  C=256: done

JA: k-means構築中...


  C=32: done


  C=64: done


  C=128: done


  C=256: done


## 2. 評価関数: assign=1 vs assign=2 vs assign=3

In [3]:
def precompute_ground_truth(embeddings, query_indices, top_k=10):
    """ground truthを事前に一括計算"""
    gt_dict = {}
    cos_all = cosine_similarity(embeddings[query_indices], embeddings)
    for i, qi in enumerate(query_indices):
        cos_all[i, qi] = -1
        gt_dict[qi] = set(np.argsort(cos_all[i])[-top_k:])
    return gt_dict


def build_multi_assign_partitions(all_sims, n_assign):
    """マルチアサインのパーティションを構築"""
    N, C = all_sims.shape
    partitions = {c: [] for c in range(C)}
    for i in range(N):
        top_pivots = np.argsort(-all_sims[i])[:n_assign]
        for pid in top_pivots:
            partitions[pid].append(i)
    for c in range(C):
        partitions[c] = np.array(partitions[c], dtype=int)
    return partitions


def evaluate_multi_assign(emb_normed, centroids, partitions, gt_dict, 
                          query_indices, n_probes, embeddings, top_k=10):
    """マルチアサインVoronoi検索の評価（事前計算GT使用）"""
    recalls = []
    candidate_counts = []
    
    for qi in query_indices:
        gt = gt_dict[qi]
        sims = centroids @ emb_normed[qi]
        top_c = np.argsort(-sims)[:n_probes]
        
        candidates_set = set()
        for c in top_c:
            candidates_set.update(partitions[c].tolist())
        candidates_set.discard(qi)
        candidates = np.array(list(candidates_set))
        candidate_counts.append(len(candidates))
        
        if len(candidates) > 0:
            cand_sims = cosine_similarity(embeddings[qi:qi+1], embeddings[candidates])[0]
            top_in_cand = candidates[np.argsort(-cand_sims)[:top_k]]
            recalls.append(len(gt & set(top_in_cand)) / top_k)
        else:
            recalls.append(0.0)
    
    return np.mean(recalls), np.mean(candidate_counts)


# クエリインデックスとground truthを事前計算
rng = np.random.default_rng(42)
query_cache = {}
for ds_name, emb in datasets.items():
    qi = rng.choice(len(emb), min(N_QUERIES, len(emb) // 2), replace=False)
    gt = precompute_ground_truth(emb, qi, TOP_K)
    query_cache[ds_name] = {'query_indices': qi, 'gt': gt}
    print(f'{ds_name}: {len(qi)} queries, GT precomputed')

EN: 200 queries, GT precomputed
JA: 200 queries, GT precomputed


## 3. assign=1 vs assign=2 vs assign=3 グリッドサーチ

In [4]:
assign_range = [1, 2, 3]
probe_range = [1, 2, 3, 4, 5, 8, 10, 15, 20]
all_results = {}

for ds_name, emb in datasets.items():
    emb_normed = kmeans_cache[ds_name]['emb_normed']
    qi = query_cache[ds_name]['query_indices']
    gt = query_cache[ds_name]['gt']
    N = len(emb)
    
    print(f'\n{"="*80}')
    print(f'{ds_name} (N={N})')
    print(f'{"="*80}')
    
    results = []
    
    for n_c in cluster_range:
        model = kmeans_cache[ds_name][n_c]
        centroids = model['centroids']
        all_sims = model['all_sims']
        
        for n_assign in assign_range:
            partitions = build_multi_assign_partitions(all_sims, n_assign)
            sizes = [len(v) for v in partitions.values()]
            
            for n_p in probe_range:
                if n_p > n_c:
                    continue
                recall, cands = evaluate_multi_assign(
                    emb_normed, centroids, partitions, gt, qi,
                    n_probes=n_p, embeddings=emb
                )
                results.append({
                    'n_clusters': n_c, 'n_assign': n_assign, 'n_probes': n_p,
                    'recall': recall, 'candidates': cands,
                    'cand_ratio': cands / N,
                    'partition_mean': np.mean(sizes),
                })
        print(f'  C={n_c}: done')
    
    all_results[ds_name] = results
    
    # C=256でassign別の結果サマリー
    for n_assign in assign_range:
        print(f'\n--- C=256, assign={n_assign} ---')
        print(f'{"P":>4} {"R@10":>8} {"候補数":>8} {"候補%":>8}')
        print('-' * 32)
        subset = [r for r in results if r['n_assign'] == n_assign 
                  and r['n_clusters'] == 256]
        for r in sorted(subset, key=lambda x: x['n_probes']):
            print(f'{r["n_probes"]:>4} {r["recall"]*100:>7.1f}% '
                  f'{r["candidates"]:>7.0f} {r["cand_ratio"]*100:>7.1f}%')


EN (N=10000)


  C=32: done


  C=64: done


  C=128: done


  C=256: done

--- C=256, assign=1 ---
   P     R@10      候補数      候補%
--------------------------------
   1    54.2%      68     0.7%
   2    67.4%     139     1.4%
   3    73.9%     206     2.1%
   4    77.8%     277     2.8%
   5    81.0%     344     3.4%
   8    86.4%     543     5.4%
  10    89.0%     676     6.8%
  15    91.9%    1008    10.1%
  20    93.4%    1331    13.3%

--- C=256, assign=2 ---
   P     R@10      候補数      候補%
--------------------------------
   1    67.9%     138     1.4%
   2    78.9%     276     2.8%
   3    83.8%     399     4.0%
   4    86.4%     528     5.3%
   5    88.5%     649     6.5%
   8    92.4%    1011    10.1%
  10    94.2%    1238    12.4%
  15    96.4%    1777    17.8%
  20    97.2%    2264    22.6%

--- C=256, assign=3 ---
   P     R@10      候補数      候補%
--------------------------------
   1    74.7%     209     2.1%
   2    84.3%     409     4.1%
   3    88.8%     581     5.8%
   4    91.1%     768     7.7%
   5    92.7%     934     9.3%
   

  C=32: done


  C=64: done


  C=128: done


  C=256: done

--- C=256, assign=1 ---
   P     R@10      候補数      候補%
--------------------------------
   1    58.4%      77     0.8%
   2    75.8%     154     1.5%
   3    82.8%     225     2.3%
   4    87.1%     291     2.9%
   5    89.7%     358     3.6%
   8    94.4%     537     5.4%
  10    95.5%     649     6.5%
  15    97.2%     917     9.2%
  20    98.1%    1185    11.9%

--- C=256, assign=2 ---
   P     R@10      候補数      候補%
--------------------------------
   1    74.2%     155     1.6%
   2    86.9%     278     2.8%
   3    90.9%     371     3.7%
   4    93.1%     458     4.6%
   5    94.4%     552     5.5%
   8    97.2%     811     8.1%
  10    98.1%     968     9.7%
  15    98.8%    1350    13.5%
  20    99.1%    1738    17.4%

--- C=256, assign=3 ---
   P     R@10      候補数      候補%
--------------------------------
   1    81.9%     228     2.3%
   2    91.0%     374     3.7%
   3    93.8%     485     4.9%
   4    95.3%     593     5.9%
   5    96.2%     713     7.1%
   

## 4. 同一候補割合での直接比較

assign=1 vs assign=2を公平に比較するため、候補割合を揃えてRecallを比較する。

In [5]:
print('='*80)
print('候補割合帯ごとの assign=1 vs assign=2 vs assign=3 比較')
print('='*80)

target_ratios = [
    (0.01, 0.03, '~2%'), (0.03, 0.07, '~5%'),
    (0.07, 0.15, '~10%'), (0.15, 0.25, '~20%'),
]

for ds_name in ['EN', 'JA']:
    results = all_results[ds_name]
    
    print(f'\n--- {ds_name} ---')
    print(f'{"帯":<8} {"assign":>7} {"Best (C,P)":>12} {"R@10":>8} {"候補数":>8} {"候補%":>8}')
    print('-' * 56)
    
    for lo, hi, label in target_ratios:
        for n_assign in assign_range:
            in_band = [r for r in results 
                      if r['n_assign'] == n_assign and lo <= r['cand_ratio'] < hi]
            if not in_band:
                continue
            best = max(in_band, key=lambda x: x['recall'])
            config = f'C={best["n_clusters"]},P={best["n_probes"]}'
            marker = ' ★' if best['recall'] == max(
                r['recall'] for r in results 
                if lo <= r['cand_ratio'] < hi
            ) else ''
            print(f'{label:<8} {n_assign:>7} {config:>12} '
                  f'{best["recall"]*100:>7.1f}% {best["candidates"]:>7.0f} '
                  f'{best["cand_ratio"]*100:>7.1f}%{marker}')
        print()

候補割合帯ごとの assign=1 vs assign=2 vs assign=3 比較

--- EN ---
帯         assign   Best (C,P)     R@10      候補数      候補%
--------------------------------------------------------
~2%            1    C=256,P=4    77.8%     277     2.8%
~2%            2    C=256,P=2    78.9%     276     2.8% ★
~2%            3    C=128,P=1    77.3%     292     2.9%

~5%            1   C=256,P=10    89.0%     676     6.8% ★
~5%            2    C=256,P=5    88.5%     649     6.5%
~5%            3    C=256,P=3    88.8%     581     5.8%

~10%           1   C=256,P=20    93.4%    1331    13.3%
~10%           2    C=128,P=8    94.4%    1337    13.4%
~10%           3    C=256,P=8    95.5%    1433    14.3% ★

~20%           1   C=128,P=20    94.6%    1839    18.4%
~20%           2   C=128,P=15    97.5%    2359    23.6%
~20%           3   C=256,P=15    98.0%    2434    24.3% ★


--- JA ---
帯         assign   Best (C,P)     R@10      候補数      候補%
--------------------------------------------------------
~2%            1   

## 5. 画像条件の完全再現: C=256, assign=2, P=2〜5

In [6]:
print('='*80)
print('画像プロジェクト条件の完全再現: C=256, assign=2')
print('='*80)

print('\n画像プロジェクト: C=256, assign=2, P=5 → 候補4.3%, BF-R@30=83.3%')

for ds_name in ['EN', 'JA']:
    results = all_results[ds_name]
    
    print(f'\n--- {ds_name} ---')
    print(f'{"assign":>7} {"P":>4} {"R@10":>8} {"候補数":>8} {"候補%":>8}')
    print('-' * 40)
    
    for n_assign in [1, 2, 3]:
        for n_p in [2, 3, 4, 5, 8, 10]:
            r = [x for x in results 
                 if x['n_clusters'] == 256 and x['n_assign'] == n_assign and x['n_probes'] == n_p]
            if r:
                r = r[0]
                marker = ' ← 画像と同条件' if n_assign == 2 and n_p == 5 else ''
                print(f'{n_assign:>7} {n_p:>4} {r["recall"]*100:>7.1f}% '
                      f'{r["candidates"]:>7.0f} {r["cand_ratio"]*100:>7.1f}%{marker}')
        print()

# NB113 assign=1との直接比較
print('\n--- assign=1(NB113) vs assign=2(本実験) 直接比較 ---')
print(f'{"条件":<30} {"EN R@10":>10} {"EN候補%":>9} {"JA R@10":>10} {"JA候補%":>9}')
print('-' * 72)
print(f'{"画像 C=256,A=2,P=5":<30} {"83.3%":>10} {"4.3%":>9} {"---":>10} {"---":>9}')

for n_assign in [1, 2]:
    for n_p in [2, 3, 5, 8, 10]:
        en_r = [x for x in all_results['EN'] 
                if x['n_clusters'] == 256 and x['n_assign'] == n_assign and x['n_probes'] == n_p]
        ja_r = [x for x in all_results['JA'] 
                if x['n_clusters'] == 256 and x['n_assign'] == n_assign and x['n_probes'] == n_p]
        if en_r and ja_r:
            label = f'C=256,A={n_assign},P={n_p}'
            print(f'{label:<30} {en_r[0]["recall"]*100:>9.1f}% '
                  f'{en_r[0]["cand_ratio"]*100:>8.1f}% '
                  f'{ja_r[0]["recall"]*100:>9.1f}% '
                  f'{ja_r[0]["cand_ratio"]*100:>8.1f}%')

画像プロジェクト条件の完全再現: C=256, assign=2

画像プロジェクト: C=256, assign=2, P=5 → 候補4.3%, BF-R@30=83.3%

--- EN ---
 assign    P     R@10      候補数      候補%
----------------------------------------
      1    2    67.4%     139     1.4%
      1    3    73.9%     206     2.1%
      1    4    77.8%     277     2.8%
      1    5    81.0%     344     3.4%
      1    8    86.4%     543     5.4%
      1   10    89.0%     676     6.8%

      2    2    78.9%     276     2.8%
      2    3    83.8%     399     4.0%
      2    4    86.4%     528     5.3%
      2    5    88.5%     649     6.5% ← 画像と同条件
      2    8    92.4%    1011    10.1%
      2   10    94.2%    1238    12.4%

      3    2    84.3%     409     4.1%
      3    3    88.8%     581     5.8%
      3    4    91.1%     768     7.7%
      3    5    92.7%     934     9.3%
      3    8    95.5%    1433    14.3%
      3   10    96.7%    1740    17.4%


--- JA ---
 assign    P     R@10      候補数      候補%
----------------------------------------
      1    

## 6. Recall目標別の最小コスト構成

各Recall目標に対して、候補数（≒Firestoreコスト）を最小化する(C, assign, P)の組み合わせ。

In [7]:
print('='*80)
print('Recall目標別 最小コスト構成')
print('='*80)

recall_targets = [0.95, 0.90, 0.85, 0.80, 0.75]

for ds_name in ['EN', 'JA']:
    results = all_results[ds_name]
    
    print(f'\n--- {ds_name} ---')
    print(f'{"目標R@10":<10} {"Best構成":<22} {"R@10":>8} {"候補数":>8} {"候補%":>8} {"IN句":>6}')
    print('-' * 66)
    
    for target in recall_targets:
        candidates = [r for r in results if r['recall'] >= target]
        if not candidates:
            print(f'{target*100:>8.0f}%  --- 達成する構成なし ---')
            continue
        best = min(candidates, key=lambda x: x['candidates'])
        config = f'C={best["n_clusters"]},A={best["n_assign"]},P={best["n_probes"]}'
        print(f'{target*100:>8.0f}%  {config:<22} {best["recall"]*100:>7.1f}% '
              f'{best["candidates"]:>7.0f} {best["cand_ratio"]*100:>7.1f}% '
              f'{best["n_probes"]:>5}')

print(f'\n※ Firestore array-contains-any は最大30要素まで')
print(f'  assign=2でもIN句の要素数はn_probes（ドキュメント側は配列フィールド）')

Recall目標別 最小コスト構成

--- EN ---
目標R@10     Best構成                     R@10      候補数      候補%    IN句
------------------------------------------------------------------
      95%  C=256,A=3,P=8             95.5%    1433    14.3%     8
      90%  C=256,A=3,P=4             91.1%     768     7.7%     4
      85%  C=256,A=2,P=4             86.4%     528     5.3%     4
      80%  C=256,A=1,P=5             81.0%     344     3.4%     5
      75%  C=256,A=2,P=2             78.9%     276     2.8%     2

--- JA ---
目標R@10     Best構成                     R@10      候補数      候補%    IN句
------------------------------------------------------------------
      95%  C=256,A=3,P=4             95.3%     593     5.9%     4
      90%  C=256,A=2,P=3             90.9%     371     3.7%     3
      85%  C=256,A=2,P=2             86.9%     278     2.8%     2
      80%  C=256,A=1,P=3             82.8%     225     2.3%     3
      75%  C=256,A=1,P=2             75.8%     154     1.5%     2

※ Firestore array-contains-

## 7. 評価・考察

### assign=2は明確にassign=1を改善する

C=256で固定し、同じprobe数での比較:

| P | EN A=1 | EN A=2 | 改善 | JA A=1 | JA A=2 | 改善 |
|---|--------|--------|------|--------|--------|------|
| 2 | 67.4% | **78.9%** | +11.5pp | 75.8% | **86.9%** | +11.1pp |
| 3 | 73.9% | **83.8%** | +9.9pp | 82.8% | **90.9%** | +8.1pp |
| 5 | 81.0% | **88.5%** | +7.5pp | 89.7% | **94.4%** | +4.7pp |
| 10 | 89.0% | **94.2%** | +5.2pp | 95.5% | **98.1%** | +2.6pp |

assign=2にすると**P=2の時点でassign=1のP=4相当のRecall**が得られる。ただし候補数も約2倍になる。

### 画像プロジェクトと同条件(C=256, A=2, P=5)での結果

| 条件 | R@10/R@30 | 候補割合 |
|------|----------|---------|
| 画像(顔認識) | 83.3% | 4.3% |
| **Text-EN** | **88.5%** | 6.5% |
| **Text-JA** | **94.4%** | 5.5% |

テキストの方が**画像よりも高いRecall**を達成。候補割合は画像の4.3%に対して5.5-6.5%とやや多いが、Recallは5〜11pp上回る。

### 候補割合を揃えた公平比較では、assignの最適値はケースバイケース

| 帯 | EN最良 | JA最良 |
|---|--------|--------|
| ~2% | A=2,P=2 (78.9%) | A=1,P=4 (87.1%) |
| ~5% | A=1,P=10 (89.0%) | A=1,P=10 (95.5%) |
| ~10% | A=3,P=8 (95.5%) | A=3,P=10 (98.8%) |
| ~20% | A=3,P=15 (98.0%) | A=3,P=20 (99.3%) |

**候補割合が同じなら、assign増加とprobe増加は等価に近い**。候補割合~5%帯ではassign=1,P=10とassign=2,P=5がほぼ同等。ただしassign=2はFirestore IN句の要素数（=P）が半分で済む利点がある。

### Recall目標別の推奨構成

| 目標 | EN構成 | EN候補% | EN IN句 | JA構成 | JA候補% | JA IN句 |
|------|--------|---------|---------|--------|---------|---------|
| R@10≥95% | C=256,A=3,P=8 | 14.3% | 8 | C=256,A=3,P=4 | 5.9% | 4 |
| R@10≥90% | C=256,A=3,P=4 | 7.7% | 4 | C=256,A=2,P=3 | 3.7% | 3 |
| R@10≥85% | C=256,A=2,P=4 | 5.3% | 4 | C=256,A=2,P=2 | 2.8% | **2** |
| R@10≥80% | C=256,A=1,P=5 | 3.4% | 5 | C=256,A=1,P=3 | 2.3% | 3 |

JA R@10≥85%が**C=256, A=2, P=2（画像と同じ構成!）**で達成可能。IN句もわずか2要素。

### assign=3の効果

assign=3はassign=2をさらに改善するが、候補数が1.5倍になるため、同じ候補割合帯で比較するとassign=2との差は小さい。ストレージ（配列要素3つ）のコスト対効果を考えるとassign=2が実用的。

### 結論

1. **assign=2は画像・テキストの両方で有効**。パーティション境界付近のドキュメントの取りこぼしを防ぐ本質的な改善
2. **テキストでもC=256, A=2, P=2〜5が実用的**。画像と同じパラメータ空間で動作する
3. **JAはテキストでもassign=2, P=2でR@10=86.9%**。画像プロジェクトの83.3%を上回る
4. **assignを増やすか、probeを増やすかは候補割合の制約次第**。Firestore IN句の制約（最大30要素）を考えると、assignを増やしてprobeを減らす方が有利
5. Firestore実装: `pivot_ids`を配列フィールド（最大2〜3要素）にして`array-contains-any`でクエリ